# Interactive Random-Wave CXL-Chain Scattering

This notebook keeps the two retained line families, `S12 = {phi1 + i phi2 = 0}` and `S13 = {phi1 + i phi3 = 0}`, as the simplest three-wave crosslinked-chain scattering test. The default case has independent `phi1`, `phi2`, and `phi3`; the `phi2`/`phi3` correlation can be tuned with one parameter.

The reduced plotting convention matches the single-chain notebook: the raw windowed intensity is divided by `int W^2 dV`, the x axis is `Q/k`, and the right panel is the explicit ratio to the local-line guide shown in the left panel.


In [ ]:
import importlib
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pyvista as pv
import numpy as np

import rw_line_network as r
import rw_line_scattering as s

r = importlib.reload(r)
s = importlib.reload(s)

print("PyVista", pv.__version__)
print("Notebook imports/reloads complete")


## Parameters

The random-wave and rendering parameters are assigned on `r`; the scattering-specific parameters are assigned on `s`.

Correlation convention for the three-wave CXL case:

- `PHI23_CORRELATION_RHO = 0` gives independent `phi2` and `phi3`.
- In the `smpl` four-field comparison, `A=L12=(phi1,phi2)` and `B=L13=(phi1,phi3)`, so `rho13=1` because `phi1` is shared and `rho24=PHI23_CORRELATION_RHO`.
- `s.SCATTERING_AMPLITUDE_METHOD = "line_segments"` is the continuous-line calculation used for high-`Q` scale checks.


In [ ]:
# Random-wave sampling settings
r.GRID_SIZE = 32
r.NUM_BLOCK = 2
r.BLOCK_OVERLAP = 1
r.RANDOM_SEED = 894894
r.NUM_MODES = (128, 128, 128)
r.K_DISTRIBUTION = "single_shell"
r.K0 = (2, 2, 2)
r.r_SIGMA_K = (0.0, 0.0, 0.0)
r.r_K_MIN = (1.0, 1.0, 1.0)
r.r_K_MAX = (1.0, 1.0, 1.0)
r.SHARED_K_VECTORS = False

# Three-wave correlation control. rho=0 is the independent default.
PHI23_CORRELATION_RHO = 0.0
if not (-1.0 < PHI23_CORRELATION_RHO <= 1.0):
    raise ValueError("PHI23_CORRELATION_RHO must be in (-1, 1].")
r.COUPLE_PHI2_PHI3 = True
r.PHI23_COUPLING_C = np.sqrt((1.0 - PHI23_CORRELATION_RHO) / (1.0 + PHI23_CORRELATION_RHO))

# Vortex tracing settings
r.USE_VORTEX_TRACING = True
r.VORTEX_FACE_PREFILTER = True
r.VORTEX_FACE_ZERO_TOL = 0.02
r.SMOOTH_VORTEX_LINES = True
r.VORTEX_SMOOTHING_SCALE = 2
r.VORTEX_TUBE_RADIUS = 0.25

# Crosslink geometry is visible, but crosslink points are not separate scatterers.
r.CROSSLINK_SEARCH_RADIUS = 1.25
r.CROSSLINK_MERGE_RADIUS = 0.5
r.CROSSLINK_BALL_RADIUS = 1.5

# Render preview settings
r.WINDOW_SIZE = (1000, 1000)
r.CAMERA_ZOOM = 0.82
r.CAMERA_AZIMUTH_DEGREES = 30.0
r.CAMERA_POLAR_DEGREES = 60.0
r.SHOW_BOUNDING_BOX = True
r.BOX_SIZE_L = None

# Scattering settings
s.LINE_SAMPLE_SPACING = 0.25
s.POINT_WEIGHT_MODE = "arclength"
s.S12_SCATTERING_WEIGHT = 1.0
s.S13_SCATTERING_WEIGHT = 1.0
s.SCATTERING_AMPLITUDE_METHOD = "line_segments"  # "line_segments" or "points"
s.SCATTERING_WINDOW = "hann_box"
s.SCATTERING_WINDOW_TAPER = 0.05
s.SUBTRACT_WINDOWED_MEAN = True
s.WINDOW_MEAN_METHOD = "numeric_1d"
s.WINDOW_NORMALIZATION = "window_squared_volume"
s.SUBTRACT_EXPLICIT_BOX_MEAN = False
s.ANALYTIC_MEAN_BUFFER_BLOCKS = 0
s.ANALYTIC_MEAN_BUFFER_MODE = "none"
s.ANALYTIC_MEAN_BUFFER_NORMALIZE_TOTAL = False
s.DYNAMIC_LINE_SAMPLE_SPACING = False
s.DYNAMIC_LINE_SAMPLE_EXPONENT = 0.5
s.DYNAMIC_LINE_SAMPLE_POWER2_SUBSETS = True
s.INCLUDE_CROSSLINK_POINTS = False
s.CROSSLINK_SCATTERING_WEIGHT = 0.0

# Real-space preview of line geometry and sampled points.
s.SHOW_STRUCTURE_LINES = True
s.SHOW_SCATTERING_SAMPLE_POINTS = True
s.SCATTERING_SAMPLE_POINTS_AS_BALLS = True
s.SCATTERING_SAMPLE_BALL_RADIUS = 0.25
s.SCATTERING_SAMPLE_POINT_COLOR = "black"
s.SCATTERING_SAMPLE_POINT_SIZE = 2.0
s.SCATTERING_SAMPLE_POINT_OPACITY = 1.0

s.Q_MIN = 0.1
s.Q_MAX = 10
s.NUM_Q = 80
s.Q_SPACING = "log"
s.NUM_Q_DIRECTIONS = 96
s.SCATTERING_SEED = None
s.INTENSITY_NORMALIZATION = "none"
s.NORMALIZE_I0 = False
s.Q_AXIS_SCALE = "mean_k"
s.NUM_SEED_AVERAGE = 8
s.STRUCTURE_SEED_START = r.RANDOM_SEED
s.STRUCTURE_SEED_STRIDE = 1009
s.SCATTERING_FLATTEN_Q_DIRECTIONS = False
s.SCATTERING_Q_CHUNK_SIZE = 10

# Gaussian conditional-sampling comparison.
MODEL_R_MIN = 1e-3
MODEL_R_MAX = 5e2
MODEL_NR = 5000
MODEL_N_SAMP = 2**15
MODEL_TAIL_START_FRACTION = 0.8
MODEL_RANDOM_SEED = 12345

print({
    "sampling": {
        "GRID_SIZE": r.GRID_SIZE,
        "NUM_BLOCK": r.NUM_BLOCK,
        "STITCHED_GRID_SIZE": r.expanded_grid_size(r.GRID_SIZE, r.NUM_BLOCK),
        "K_DISTRIBUTION": r.K_DISTRIBUTION,
        "K0": r.K0,
        "PHI23_CORRELATION_RHO": PHI23_CORRELATION_RHO,
        "PHI23_COUPLING_C": r.PHI23_COUPLING_C,
    },
    "scattering": {
        "LINE_SAMPLE_SPACING": s.LINE_SAMPLE_SPACING,
        "POINT_WEIGHT_MODE": s.POINT_WEIGHT_MODE,
        "SCATTERING_AMPLITUDE_METHOD": s.SCATTERING_AMPLITUDE_METHOD,
        "SCATTERING_WINDOW": s.SCATTERING_WINDOW,
        "SUBTRACT_WINDOWED_MEAN": s.SUBTRACT_WINDOWED_MEAN,
        "INTENSITY_NORMALIZATION": s.INTENSITY_NORMALIZATION,
        "NUM_Q_DIRECTIONS": s.NUM_Q_DIRECTIONS,
        "STRUCTURE_SEEDS": s.structure_seed_values(),
        "SCATTERING_Q_CHUNK_SIZE": s.SCATTERING_Q_CHUNK_SIZE,
    },
    "gaussian_four_field_model": {
        "rho13": 1.0,
        "rho24": PHI23_CORRELATION_RHO,
        "MODEL_R_MIN": MODEL_R_MIN,
        "MODEL_R_MAX": MODEL_R_MAX,
        "MODEL_NR": MODEL_NR,
        "MODEL_N_SAMP": MODEL_N_SAMP,
        "MODEL_TAIL_START_FRACTION": MODEL_TAIL_START_FRACTION,
        "MODEL_RANDOM_SEED": MODEL_RANDOM_SEED,
    },
})


## Build CXL-Chain Structure

The full retained structure contains both `S12` and `S13`. Single-family copies are used only for diagnostics and density normalization.


In [ ]:
print("[build] building retained two-family CXL structure for preview/reference seed...")
t_build0 = time.perf_counter()
structure = s.build_line_structure(
    line_sample_spacing=s.LINE_SAMPLE_SPACING,
    include_crosslinks=s.INCLUDE_CROSSLINK_POINTS,
    print_timing=True,
    timing_label="preview seed",
)
s12_structure = s.as_single_s12_structure(structure, line_sample_spacing=s.LINE_SAMPLE_SPACING)
s13_structure = s.as_single_s13_structure(structure, line_sample_spacing=s.LINE_SAMPLE_SPACING)
print(f"[build] structure finished in {time.perf_counter() - t_build0:.2f}s")

print("retained sampled scattering points:", len(structure.points))
print("retained continuous line segments:", len(structure.segment_starts))
print("S12 line points/lines:", structure.s12_lines.n_points, structure.s12_lines.n_lines)
print("S13 line points/lines:", structure.s13_lines.n_points, structure.s13_lines.n_lines)
print("S12 segments:", len(s12_structure.segment_starts))
print("S13 segments:", len(s13_structure.segment_starts))


## Real-Space Preview


In [ ]:
plotter = s.make_structure_plotter(structure)
plotter.show(jupyter_backend="static")


## Scattering Curve And Four-Field Gaussian Reference


In [ ]:
def status(message):
    print(message, flush=True)


def load_gaussian_sampling_model_module():
    import importlib.util
    import sys
    from pathlib import Path
    model_path = Path("smpl") / "rw_line_scattering.py"
    if not model_path.exists():
        model_path = Path.cwd() / "smpl" / "rw_line_scattering.py"
    status(f"[model] loading four-field Gaussian module from {model_path}...")
    t0 = time.perf_counter()
    spec = importlib.util.spec_from_file_location("rw_line_scattering_smpl", model_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    status(f"[model] module loaded in {time.perf_counter() - t0:.2f}s")
    return module


def gaussian_four_field_reference(q_over_k, rho24):
    status(
        f"[model] computing four-field reference: rho13=1, rho24={rho24}, "
        f"Nr={MODEL_NR}, Q={len(q_over_k)}, N_samp={MODEL_N_SAMP}"
    )
    t0 = time.perf_counter()
    ris = load_gaussian_sampling_model_module()
    k0_model = 1.0
    r_grid = np.linspace(MODEL_R_MIN, MODEL_R_MAX, MODEL_NR)
    W_tail = ris.tail_window(r_grid, MODEL_TAIL_START_FRACTION*MODEL_R_MAX, MODEL_R_MAX)
    xi12 = ris.make_qmc_normals(12, MODEL_N_SAMP, MODEL_RANDOM_SEED)

    self_result = ris.compute_self_correlation(r_grid, k0_model, xi12, progress=True)
    rho0_model = k0_model**2 / (3.0*np.pi)
    C_self_coherent = self_result.C - rho0_model**2
    I_AA_model = ris.hankel_transform(r_grid, C_self_coherent * W_tail, q_over_k)

    cross_result = ris.compute_cross_correlation(
        r_grid,
        k0_model,
        1.0,
        rho24,
        xi12,
        progress=True,
    )
    C_cross_coherent = cross_result.C - rho0_model**2
    I_AB_model = ris.hankel_transform(r_grid, C_cross_coherent * W_tail, q_over_k)
    I_total_model = 2*I_AA_model + 2*I_AB_model
    status(f"[model] four-field reference complete in {time.perf_counter() - t0:.2f}s")
    return {
        "r_grid": r_grid,
        "W_tail": W_tail,
        "rho0_model": rho0_model,
        "C_self": self_result.C,
        "C_self_coherent": C_self_coherent,
        "C_cross": cross_result.C,
        "C_cross_coherent": C_cross_coherent,
        "I_AA_model": I_AA_model,
        "I_AB_model": I_AB_model,
        "I_total_model": I_total_model,
        "M_self": self_result.M,
        "M_cross": cross_result.M,
    }


t_cell0 = time.perf_counter()
status("[main] starting CXL-chain scattering and comparison cell")
seeds = s.structure_seed_values()
(
    q,
    iq_raw,
    iq_raw_std,
    iq_box,
    point_counts,
    full_window_diagnostics,
    s12_window_diagnostics,
    s13_window_diagnostics,
) = s.compute_seed_averaged_cxl_chain_scattering(seeds, print_timing=True)

status("[main] computing Q-axis metadata and reduced units...")
k_mean = float(np.mean(r._field_parameter_values(r.K0, "K0")))
wave_coordinate_scale = float(r.GRID_SIZE)
box_grid_side = float(r.GRID_SIZE - 1)
k_wave_angular_mean = 2.0 * np.pi * k_mean
k_angular_grid = k_wave_angular_mean / wave_coordinate_scale
box_l = r.BOX_SIZE_L if r.BOX_SIZE_L is not None else float(r.expanded_grid_size(r.GRID_SIZE, r.NUM_BLOCK) - 1)
box_volume_grid = box_l**3
window_volume = s.window_volume()
window_squared_volume = s.window_squared_volume()

full_d_w2_values = np.asarray([diag["density_from_w2"] for diag in full_window_diagnostics], dtype=float)
s12_d_w2_values = np.asarray([diag["density_from_w2"] for diag in s12_window_diagnostics], dtype=float)
s13_d_w2_values = np.asarray([diag["density_from_w2"] for diag in s13_window_diagnostics], dtype=float)
d12 = float(np.nanmean(s12_d_w2_values))
d13 = float(np.nanmean(s13_d_w2_values))
d_total = d12 + d13
if not np.isfinite(d_total) or d_total <= 0.0:
    d_total = float(np.nanmean(full_d_w2_values))
    d12 = 0.5*d_total
    d13 = 0.5*d_total
d_theory_single = k_angular_grid**2 / (3.0*np.pi)
d_theory_total = 2*d_theory_single
N = d_total**2
q_grid_units = q
q_over_k = q / k_angular_grid
q_scaled = q_over_k
q_axis_label = r"$Q/k$"

I_windowed = iq_raw / window_squared_volume
I_windowed_std = iq_raw_std / window_squared_volume
I_reduced = I_windowed / N
I_reduced_std = I_windowed_std / N
line_asymptote_reduced = np.pi / np.maximum(q_grid_units * d_total, np.finfo(float).tiny)
line_ratio_reduced = I_reduced / line_asymptote_reduced
line_ratio_reduced_std = I_reduced_std / line_asymptote_reduced

model = gaussian_four_field_reference(q_over_k, PHI23_CORRELATION_RHO)
rho0_model = model["rho0_model"]
d_gaussian_theory = k_angular_grid**2 * rho0_model
scale_A = d12 / d_gaussian_theory
scale_B = d13 / d_gaussian_theory
scale_AB = np.sqrt(max(d12*d13, 0.0)) / d_gaussian_theory
I_AA_gaussian = k_angular_grid * model["I_AA_model"] * scale_A
I_BB_gaussian = k_angular_grid * model["I_AA_model"] * scale_B
I_AB_gaussian = k_angular_grid * model["I_AB_model"] * scale_AB
I_gaussian_windowed = I_AA_gaussian + I_BB_gaussian + 2*I_AB_gaussian
I_gaussian_reduced = I_gaussian_windowed / N
line_ratio_gaussian_reduced = I_gaussian_reduced / line_asymptote_reduced

status("reduced normalization calculation:")
status(f"  K = 2*pi*<k_cycles>/GRID_SIZE = {k_angular_grid:.6g}")
status(f"  rho24 = corr(phi2, phi3) = {PHI23_CORRELATION_RHO:.6g}")
status(f"  smpl mapping: rho13=1, rho24={PHI23_CORRELATION_RHO:.6g}")
status(f"  theoretical single-line density = {d_theory_single:.6g}")
status(f"  theoretical two-family density = {d_theory_total:.6g}")
status(f"  measured d12 = {d12:.6g}")
status(f"  measured d13 = {d13:.6g}")
status(f"  d_total used for scaling = {d_total:.6g}")
status(f"  N = d_total^2 = {N:.6g}")
status(f"  window-adjusted intensity I(Q) = I_raw(Q)/int(W^2 dV)")
status(f"  left-panel local-line guide: I(Q)/d_total^2 = pi/(Q*d_total)")
status(f"  right-panel ratio: [I(Q)/d_total^2] / [pi/(Q*d_total)] should approach 1")
status("window diagnostics:")
status(f"  int W dV = {window_volume:.6g}")
status(f"  int W^2 dV = {window_squared_volume:.6g}")
status(f"  full density from W^2 integral = {np.nanmean(full_d_w2_values):.6g}")
status(f"[main] scattering/comparison cell complete in {time.perf_counter() - t_cell0:.2f}s")


In [ ]:
print("[plot] rendering scattering comparison plot...")
t_plot0 = time.perf_counter()
fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.2), constrained_layout=True)
ax = axes[0]

ax.plot(q_scaled, I_reduced, color="black", lw=1.6, label="Trajectory")
if len(seeds) > 1:
    ax.fill_between(q_scaled, I_reduced - I_reduced_std, I_reduced + I_reduced_std, color="black", alpha=0.18, linewidth=0)
ax.plot(q_over_k, I_gaussian_reduced, color="tab:orange", lw=1.5, ls="--", label="Gaussian four-field")
ax.plot(q_scaled, line_asymptote_reduced, color="tab:blue", lw=1.2, ls=":", label=r"$\pi/(Qd)$")
ax.set_xlabel(q_axis_label)
ax.set_ylabel(r"$I(Q)/d^2$")
ax.set_xscale("log")
ax.set_yscale("log")
q_half_box = 4.0 * np.pi / box_l
q_half_box_scaled = q_half_box / k_angular_grid
ax.axvline(q_half_box_scaled, color="tab:red", lw=1.2, ls="--", alpha=0.8, label=r"$2\pi/(L/2)$")
ax.legend(frameon=False)
ax.grid(True, alpha=0.25)

ax = axes[1]
ax.plot(q_scaled, line_ratio_reduced, color="black", lw=1.6, label="Trajectory")
if len(seeds) > 1:
    ax.fill_between(q_scaled, line_ratio_reduced - line_ratio_reduced_std, line_ratio_reduced + line_ratio_reduced_std, color="black", alpha=0.18, linewidth=0)
ax.plot(q_over_k, line_ratio_gaussian_reduced, color="tab:orange", lw=1.5, ls="--", label="Gaussian four-field")
ax.axhline(1.0, color="tab:blue", lw=1.2, ls=":", label=r"$[\pi/(Qd)]/[\pi/(Qd)]$")
ax.axvline(q_half_box_scaled, color="tab:red", lw=1.2, ls="--", alpha=0.8)
ax.set_xlabel(q_axis_label)
ax.set_ylabel(r"$[I(Q)/d^2] / [\pi/(Qd)]$")
ax.set_xscale("log")
ax.legend(frameon=False)
ax.grid(True, alpha=0.25)

print("half-box marker Q/k:", q_half_box_scaled)
print("seeds:", seeds)
print("point counts:", point_counts)
print("rho24:", PHI23_CORRELATION_RHO)
print("K angular per grid unit:", k_angular_grid)
print("d12, d13, d_total:", d12, d13, d_total)
print("theory single, theory total:", d_theory_single, d_theory_total)
print("int W^2 dV:", window_squared_volume)
print("raw I(Q) range:", float(iq_raw.min()), float(iq_raw.max()))
print("reduced I/d^2 range:", float(I_reduced.min()), float(I_reduced.max()))
print("line-guide ratio range:", float(line_ratio_reduced.min()), float(line_ratio_reduced.max()))
print(f"[plot] plot cell complete in {time.perf_counter() - t_plot0:.2f}s")


## Save Data


In [ ]:
print("[save] saving scattering outputs...")
t_save0 = time.perf_counter()
out_path = s.save_scattering_curve(
    "rw_cxl_chain_scattering_iq.csv",
    q,
    iq_raw,
    std=iq_raw_std,
    box_intensity=iq_box,
)
print(out_path)

np.savez_compressed(
    "rw_cxl_chain_scattering_compare.npz",
    q=q,
    q_over_k=q_over_k,
    q_scaled=q_scaled,
    iq_raw=iq_raw,
    iq_raw_std=iq_raw_std,
    I_windowed=I_windowed,
    I_windowed_std=I_windowed_std,
    I_reduced=I_reduced,
    I_reduced_std=I_reduced_std,
    line_ratio_reduced=line_ratio_reduced,
    line_ratio_reduced_std=line_ratio_reduced_std,
    I_gaussian_windowed=I_gaussian_windowed,
    I_gaussian_reduced=I_gaussian_reduced,
    line_ratio_gaussian_reduced=line_ratio_gaussian_reduced,
    I_AA_gaussian=I_AA_gaussian,
    I_BB_gaussian=I_BB_gaussian,
    I_AB_gaussian=I_AB_gaussian,
    k_angular_grid=k_angular_grid,
    d12=d12,
    d13=d13,
    d_total=d_total,
    d_theory_single=d_theory_single,
    d_theory_total=d_theory_total,
    N=N,
    rho24=PHI23_CORRELATION_RHO,
    rho13=1.0,
    window_volume=window_volume,
    window_squared_volume=window_squared_volume,
    line_asymptote_reduced=line_asymptote_reduced,
    point_counts=np.asarray(point_counts),
    seeds=np.asarray(seeds),
)
print("rw_cxl_chain_scattering_compare.npz")
print(f"[save] save cell complete in {time.perf_counter() - t_save0:.2f}s")
